In [ ]:
# 1. Configuracion

from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA  = "si7006_t2"
TBL_SILVER       = f"{CATALOG}.{SCHEMA}.orders_silver"
TBL_GOLD_ALERTAS = f"{CATALOG}.{SCHEMA}.gold_alertas_reorder"
VOL_CHECKPOINTS = f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints"
CKPT_GOLD_ALERTAS = f"{VOL_CHECKPOINTS}/gold_alertas_reorder"

WINDOW_SIZE = "2 minutes"
WATERMARK_DELAY = "30 seconds"
UNITS_THRESHOLD = 12
ORDERS_THRESHOLD = 2

print(f"Silver:       {TBL_SILVER}")
print(f"Gold alertas: {TBL_GOLD_ALERTAS}")
print(f"Checkpoint:   {CKPT_GOLD_ALERTAS}")
print(f"Window size:  {WINDOW_SIZE}")
print(f"Watermark:    {WATERMARK_DELAY}")
print(f"Thresholds:   units>={UNITS_THRESHOLD}, orders>={ORDERS_THRESHOLD}")


In [ ]:
# 2. Lectura streaming desde Silver + agregacion por ventana

df_gold_alertas = (
    spark.readStream
        .format("delta")
        .table(TBL_SILVER)
        .withWatermark("event_time", WATERMARK_DELAY)
        .groupBy(
            F.window("event_time", WINDOW_SIZE).alias("w"),
            F.col("stock_code"),
            F.col("country")
        )
        .agg(
            F.approx_count_distinct("invoice_no").alias("n_orders"),
            F.sum("quantity").alias("units_sold"),
            F.round(F.sum("total_amount"), 2).alias("revenue")
        )
        .withColumn(
            "is_alert",
            (F.col("units_sold") >= F.lit(UNITS_THRESHOLD))
            & (F.col("n_orders") >= F.lit(ORDERS_THRESHOLD))
        )
        .select(
            F.col("w.start").alias("window_start"),
            F.col("w.end").alias("window_end"),
            "stock_code",
            "country",
            "n_orders",
            "units_sold",
            "revenue",
            "is_alert"
        )
)

df_gold_alertas.printSchema()


In [ ]:
# 3. Loop de escritura con Trigger.AvailableNow

import time

ITERACIONES = 5
PAUSA_SEGUNDOS = 12

for i in range(1, ITERACIONES + 1):
    query = (
        df_gold_alertas.writeStream
            .format("delta")
            .outputMode("append")
            .option("checkpointLocation", CKPT_GOLD_ALERTAS)
            .option("mergeSchema", "false")
            .trigger(availableNow=True)
            .toTable(TBL_GOLD_ALERTAS)
    )
    query.awaitTermination()

    total = spark.sql(f"SELECT COUNT(*) AS n FROM {TBL_GOLD_ALERTAS}").collect()[0]["n"]
    print(f"[{i:02d}/{ITERACIONES}] {time.strftime('%H:%M:%S')} | total: {total:,}")

    if i < ITERACIONES:
        time.sleep(PAUSA_SEGUNDOS)


In [ ]:
# 4. Validacion rapida

spark.table(TBL_GOLD_ALERTAS).printSchema()

spark.sql(f"""
    SELECT COUNT(*) AS filas,
           COUNT(DISTINCT window_start) AS ventanas,
           ROUND(SUM(revenue), 2) AS revenue_total,
           SUM(CASE WHEN is_alert THEN 1 ELSE 0 END) AS alertas_activas
    FROM {TBL_GOLD_ALERTAS}
""").show(truncate=False)

spark.sql(f"""
    SELECT is_alert,
           COUNT(*) AS grupos,
           SUM(units_sold) AS units_total,
           ROUND(SUM(revenue), 2) AS revenue_total
    FROM {TBL_GOLD_ALERTAS}
    GROUP BY is_alert
    ORDER BY is_alert DESC
""").show(truncate=False)

spark.sql(f"SELECT * FROM {TBL_GOLD_ALERTAS} ORDER BY window_start DESC, units_sold DESC LIMIT 20").show(truncate=False)


In [ ]:
# 5. Historial Delta

spark.sql(f"DESCRIBE HISTORY {TBL_GOLD_ALERTAS}").select(
    "version", "timestamp", "operation",
    "operationMetrics.numOutputRows"
).show(truncate=False)
